In [1]:
import stim
import pymatching
import numpy as np
from itertools import combinations
from collections import deque
import time

def exp(W,T, d, p):
    d = d                     # code distance
    T = T                     # total QEC cycles
    W = W                     # LILLIPUT window size
    p = p                     # physical error rate (0.01 - 0.02)
    MAX_SYNDROME_WEIGHT = 2   # LUT cutoff weight

    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_x",
        before_measure_flip_probability=p,
        distance=d,
        rounds=T,
        #before_round_data_depolarization=p
    )

    dem = circuit.detector_error_model(decompose_errors=True)
    matching = pymatching.Matching.from_detector_error_model(dem)

    num_det = dem.num_detectors
    sampler = circuit.compile_detector_sampler()

    lut = {}

    for w in range(MAX_SYNDROME_WEIGHT + 1):
        for locs in combinations(range(num_det), w):
            syndrome = np.zeros(num_det, dtype=np.uint8)
            for i in locs:
                syndrome[i] = 1

            correction = matching.decode(syndrome, algorithm="mwpm")
            lut[tuple(syndrome.tolist())] = correction

#print(f"LUT size: {len(lut)}")


    shots = 1000
    logical_errors = 0
    t_start = time.time()
    for _ in range(shots):
        dets, obs = sampler.sample(shots=1, separate_observables=True)
        dets = dets[0]
        logical_flip_true = obs[0][0]

        window = deque(maxlen=W)
        frame_prediction = 0

        for t in range(num_det):
            window.append((t, dets[t]))

            if len(window) < W and t < num_det - W:
                continue

            full_syndrome = np.zeros(num_det, dtype=np.uint8)
            for idx, bit in window:
                full_syndrome[idx] = bit

            key = tuple(full_syndrome.tolist())

            if key in lut:
                pred = lut[key]
            else:
                pred = matching.decode(full_syndrome, algorithm="union-find")

            oldest_idx, _ = window[0]
            frame_prediction ^= pred[0]

            window.popleft()

        if frame_prediction != logical_flip_true:
            logical_errors += 1
        t_end = time.time()
        
    return logical_errors/shots*100, t_end - t_start 
#print("Logical error rate (stable sliding-window LILLIPUT):", logical_errors / shots*100)


In [2]:
exp(3,10, 5, 0.01)
#logical_errors and time taken

(6.1, 1.1805121898651123)